<a href="https://colab.research.google.com/github/9sushant/Project1/blob/main/run_finetuning_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Mistral QLoRA Fine-tuning in Google Colab

This notebook provides the environment to run the fine-tuning script (`finetune.py`) from your GitHub repository.

**Pre-requisites:**
1. Go to **Runtime -> Change runtime type** and select **T4 GPU** or better.
2. Ensure you have a Hugging Face token and have accepted the terms for the Mistral model on the Hugging Face hub.

In [1]:
# Step 1: Clone the repository and navigate into the folder
!git clone https://github.com/9sushant/Project1.git
%cd Project1

Cloning into 'Project1'...
remote: Enumerating objects: 60, done.
remote: Counting objects: 100% (60/60), done.
remote: Compressing objects: 100% (44/44), done.
remote: Total 60 (delta 31), reused 43 (delta 16), pack-reused 0 (from 0)
Receiving objects: 100% (60/60), 95.55 KiB | 949.00 KiB/s, done.
Resolving deltas: 100% (31/31), done.
/content/Project1


In [2]:
# Step 2: Install dependencies
!pip install -r requirements.txt
!pip install -U bitsandbytes transformers peft accelerate trl datasets huggingface_hub

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 528.8/528.8 kB 8.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 13.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 53.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 383.7/383.7 kB 28.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 520.7/520.7 kB 46.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 612.9/612.9 kB 50.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 93.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 12.3 MB/s eta 0:00:00
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 18.1.0
    Uninstalling pyarrow-18.1.0:
      Successfully uninstalled pyarrow-18.1.0
  Attempting uninstall: hf-xet
    Found existing installation: hf-xet 1.3.1
    Uninstalling hf-xet-1.3.1:
      Successfully uninstalled hf-xet-1.3.1
  Attempting uninstall: huggingface_hub
    Found ex

In [3]:
# Step 3: Login to Hugging Face (to download the Mistral base model)
# You will need to click the link that appears and paste your access token.
from huggingface_hub import notebook_login
notebook_login()

In [4]:
!git pull


Already up to date.


In [5]:
# Step 4: Run the fine-tuning script!
!python finetune.py

Loading Model: mistralai/Mistral-7B-Instruct-v0.2
Generating train split: 500 examples [00:00, 181242.07 examples/s]
Map: 100% 500/500 [00:00<00:00, 80759.09 examples/s]
config.json: 100% 596/596 [00:00<00:00, 3.58MB/s]
tokenizer_config.json: 2.10kB [00:00, 4.97MB/s]
tokenizer.json: 1.80MB [00:00, 108MB/s]
tokenizer.model: 100% 493k/493k [00:01<00:00, 349kB/s] 
special_tokens_map.json: 100% 414/414 [00:00<00:00, 2.36MB/s]
`torch_dtype` is deprecated! Use `dtype` instead!
model.safetensors.index.json: 25.1kB [00:00, 73.7MB/s]
Fetching 3 files: 100% 3/3 [10:25<00:00, 208.47s/it]
Download complete: 100% 14.5G/14.5G [10:25<00:00, 23.2MB/s]
Loading weights: 100% 291/291 [00:59<00:00,  4.90it/s]
generation_config.json: 100% 111/111 [00:00<00:00, 668kB/s]
trainable params: 83,886,080 || all params: 7,325,618,176 || trainable%: 1.1451
Adding EOS to train dataset: 100% 500/500 [00:00<00:00, 15576.81 examples/s]
Tokenizing train dataset: 100% 500/500 [00:00<00:00, 2771.04 examples/s]
Truncating 

In [ ]:
!git pull

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel

base_model_name = "mistralai/Mistral-7B-Instruct-v0.2"
adapter_path = "./mistral-finetuned"

tokenizer = AutoTokenizer.from_pretrained(base_model_name, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

model = AutoModelForCausalLM.from_pretrained(
    base_model_name,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.float16,
    trust_remote_code=True
)

model = PeftModel.from_pretrained(model, adapter_path)
model.eval()
print("✅ Fine-tuned model loaded!")


Already up to date.


`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

In [ ]:
def chat(question, max_new_tokens=256):
    prompt = f"<s>[INST] {question} [/INST]"
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            repetition_penalty=1.15
        )
    response = tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
    return response.strip()

print("✅ Chat function ready!")


In [ ]:
test_questions = [
    "Buddhi, Bhojan aur Kaal (Mritayu) ki utpatti ka kram kya hai?",
    "Bhojan aur neend ka aapas mein kya sambandh bataya gaya hai?",
    "Yogiraj ne is sthiti ko kya naam diya hai aur isko sach mein kaun samajh sakta hai?",
]

for i, q in enumerate(test_questions, 1):
    print(f"\n{'='*60}")
    print(f"Q{i}: {q}")
    print(f"{'='*60}")
    print(f"A: {chat(q)}")


Once training is complete, the adapters are saved in `./mistral-finetuned`.
You can compress this folder and download it, or push it to Hugging Face Hub!